In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import os

In [2]:
spark = (SparkSession.builder
         .appName("ClickHouseSpark")
         .master("local[*]")
         .config("spark.jars.packages", "com.clickhouse:clickhouse-jdbc:0.8.0,org.postgresql:postgresql:42.7.3")
         .getOrCreate())

In [3]:
clickhouse_host = "clickhouse"
clickhouse_port = "8123"
clickhouse_user = os.getenv("CLICKHOUSE_USER", "default")
clickhouse_password = os.getenv("CLICKHOUSE_PASSWORD")
clickhouse_db = os.getenv("CLICKHOUSE_DB", "default")

clickhouse_url = f"jdbc:clickhouse://{clickhouse_host}:{clickhouse_port}/{clickhouse_db}"

base_reader_clickhouse = (spark.read
               .format("jdbc")
               .option("url", clickhouse_url)
               .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
               .option("user", clickhouse_user)
               .option("password", clickhouse_password))

In [4]:
def write_to_clickhouse(df, table):
    (df.write
       .format("jdbc")
       .option("url", clickhouse_url)
       .option("batchsize", "100000")
       .option("socket_timeout", "300000")
       .option("rewriteBatchedStatements", "true")
       .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
       .option("user", clickhouse_user)
       .option("password", clickhouse_password)
       .option("dbtable", table)
       .mode("append")
       .save())


In [5]:
pg_user = os.getenv("POSTGRES_USER", "postgres")
pg_password = os.getenv("POSTGRES_PASSWORD")
pg_host = os.getenv("POSTGRES_HOST", "postgres")
pg_port = os.getenv("POSTGRES_PORT", 5432)
pg_db = "postgres"

pg_url = f"jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}"

base_reader_pg = (spark.read
               .format("jdbc")
               .option("url", pg_url)
               .option("driver", "org.postgresql.Driver")
               .option("user", pg_user)
               .option("password", pg_password))

# Data Mart №1. Продукты.
Цель: Анализ выручки, количества продаж и популярности продуктов.

- Топ-10 самых продаваемых продуктов.
- Общая выручка по категориям продуктов.
- Средний рейтинг и количество отзывов для каждого продукта.

In [6]:
dim_products = base_reader_pg.option('dbtable', 'public.dim_products').load()

In [7]:
dim_products.show(5)

+---+------------+----------------+-------------+----------------+--------------+-------------+------------+-------------+----------------+--------------------+--------------+---------------+--------------------+-------------------+-------------------+
| id|product_name|product_category|product_price|product_quantity|product_weight|product_color|product_size|product_brand|product_material| product_description|product_rating|product_reviews|product_release_date|product_expiry_date|product_supplier_id|
+---+------------+----------------+-------------+----------------+--------------+-------------+------------+-------------+----------------+--------------------+--------------+---------------+--------------------+-------------------+-------------------+
|  1|    Dog Food|            Food|        77.97|              89|          13.4|       Indigo|      Medium|        Skajo|           Steel|Aliquam quis turp...|           2.1|             97|          10/19/2011|         10/21/2028|         

In [8]:
fact_sales = base_reader_pg.option('dbtable', 'public.fact_sales').load()

In [9]:
fact_sales.show(5)

+---+----------+----------------+--------------+---------------+-------------+----------------+-------------+----------------+
| id| sale_date|sale_customer_id|sale_seller_id|sale_product_id|sale_store_id|sale_supplier_id|sale_quantity|sale_total_price|
+---+----------+----------------+--------------+---------------+-------------+----------------+-------------+----------------+
|  1| 5/14/2021|               1|             1|              1|            1|               1|            4|           487.7|
|  2|11/13/2021|               2|             2|              2|            2|               2|           10|          484.61|
|  3| 12/4/2021|               3|             3|              3|            3|               3|            9|          144.24|
|  4| 8/10/2021|               4|             4|              4|            4|               4|            4|          441.09|
|  5|  2/4/2021|               5|             5|              5|            5|               5|            6|  

In [10]:
product_sales = fact_sales.groupBy('sale_product_id').agg(
    count('sale_product_id').alias('total_sales'),
    sum('sale_total_price').alias('total_revenue')
).withColumnRenamed('sale_product_id', 'id')

In [11]:
datamart_products = dim_products.join(product_sales, 'id').withColumns({
    'avg_rating': avg('product_rating').over(Window.partitionBy('id')),
    'avg_reviews': avg('product_reviews').over(Window.partitionBy('id')),
    'category_revenue': sum('total_sales').over(Window.partitionBy('product_category'))
}).select('id', 'product_category', 'total_sales', 'total_revenue', 'avg_rating', 'avg_reviews', 'category_revenue').orderBy(desc('total_sales'), 'id')

In [12]:
datamart_products.show(5)

+---+----------------+-----------+------------------+------------------+-----------+----------------+
| id|product_category|total_sales|     total_revenue|        avg_rating|avg_reviews|category_revenue|
+---+----------------+-----------+------------------+------------------+-----------+----------------+
|  1|            Food|          1|487.70001220703125|2.0999999046325684|       97.0|            3267|
|  2|            Food|          1| 484.6099853515625|2.4000000953674316|       37.0|            3267|
|  3|            Food|          1|144.24000549316406|               3.0|      218.0|            3267|
|  4|            Food|          1| 441.0899963378906| 2.200000047683716|      959.0|            3267|
|  5|            Cage|          1|  72.2699966430664|               4.5|      130.0|            3327|
+---+----------------+-----------+------------------+------------------+-----------+----------------+
only showing top 5 rows



In [13]:
write_to_clickhouse(datamart_products, 'datamart_products')

# Data Mart №2. Клиенты.
Цель: Анализ покупательского поведения и сегментация клиентов.

- Топ-10 клиентов с наибольшей общей суммой покупок.
- Распределение клиентов по странам.
- Средний чек для каждого клиента.


In [14]:
dim_customers = base_reader_pg.option("dbtable", "dim_customers").load()

In [15]:
dim_customers.show(5)

+---+-------------------+------------------+------------+--------------------+--------------------+-----------------+-----------------+------------------+------------+---------------+----------+
| id|customer_first_name|customer_last_name|customer_age|      customer_email|customer_postal_code|customer_pet_type|customer_pet_name|customer_pet_breed|pet_category|pet_category_id|country_id|
+---+-------------------+------------------+------------+--------------------+--------------------+-----------------+-----------------+------------------+------------+---------------+----------+
|  1|             Barron|           Rawlyns|          61|bmassingham0@army...|                NULL|              cat|        Priscella|Labrador Retriever|        NULL|              2|        43|
|  2|                Ham|          Knowller|          78|  cscudder1@time.com|              73-115|             bird|          Dalenna|Labrador Retriever|        NULL|              2|       165|
|  3|           Farleigh|

In [16]:
total_spend = fact_sales.groupby('sale_customer_id').sum('sale_total_price').withColumnsRenamed(
    {'sum(sale_total_price)': 'total_spend',
     'sale_customer_id': 'id'})

In [17]:
country_distribution = dim_customers.groupBy('country_id').count().withColumnRenamed('count', 'country_distribution')

In [18]:
avg_spend = fact_sales.groupBy('sale_customer_id').avg('sale_total_price').withColumnsRenamed(
    {'avg(sale_total_price)': 'avg_spend',
     'sale_customer_id': 'id'})

In [19]:
datamart_customers = (dim_customers.join(total_spend.withColumnRenamed('sale_customer_id', 'id'), 'id')
              .join(country_distribution, 'country_id')
              .join(avg_spend, 'id')
              .select('id', 'total_spend', 'avg_spend', 'country_id', 'country_distribution')
              .orderBy(desc('total_spend'), desc('avg_spend'), 'id'))

In [20]:
datamart_customers.show(5)

+----+------------------+------------------+----------+--------------------+
|  id|       total_spend|         avg_spend|country_id|country_distribution|
+----+------------------+------------------+----------+--------------------+
|1188| 499.8500061035156| 499.8500061035156|         3|                  46|
|6422|499.79998779296875|499.79998779296875|       166|                 336|
|6361|  499.760009765625|  499.760009765625|        43|                1738|
|9923|  499.760009765625|  499.760009765625|        94|                1174|
|5616| 499.7300109863281| 499.7300109863281|       165|                 332|
+----+------------------+------------------+----------+--------------------+
only showing top 5 rows



In [21]:
write_to_clickhouse(datamart_customers, 'datamart_customers')

# Data Mart №3. Продажи по времени.
Цель: Анализ сезонности и трендов продаж.
- Месячные и годовые тренды продаж.
- Сравнение выручки за разные периоды.
- Средний размер заказа по месяцам.

In [22]:
fact_sales

DataFrame[id: int, sale_date: string, sale_customer_id: int, sale_seller_id: int, sale_product_id: int, sale_store_id: int, sale_supplier_id: int, sale_quantity: int, sale_total_price: float]

In [23]:
temp_df = fact_sales.withColumn('temp_date', to_date('sale_date', 'M/d/yyyy'))

seasonal_sales = temp_df.withColumns({
    'sale_year': year('temp_date'),
    'sale_month': month('temp_date'),
}).drop('temp_date')

In [24]:
datamart_seasonal = seasonal_sales.groupBy('sale_year', 'sale_month').agg(
    count('id').alias('total_orders'),
    sum('sale_total_price').alias('monthly_revenue'),
    avg('sale_quantity').alias('avg_order_volume')
).orderBy('sale_year', 'sale_month')

In [25]:
write_to_clickhouse(datamart_seasonal, 'datamart_seasonal')

# Data Mart №4. Магазины.
Цель: Анализ эффективности магазинов.
- Топ-5 магазинов с наибольшей выручкой.
- Распределение продаж по городам и странам.
- Средний чек для каждого магазина.

In [26]:
dim_stores = base_reader_pg.option("dbtable", "dim_stores").load()

In [27]:
window_store_sales = Window.partitionBy('id')

In [28]:
datamart_stores = (dim_stores
    .join(fact_sales
    .select('sale_store_id', 'sale_total_price')
    .withColumnRenamed('sale_store_id', 'id'), 'id')
    .withColumns({
        'store_total_sales': sum('sale_total_price').over(window_store_sales),
        'store_city_distribution': count('id').over(Window.partitionBy('store_city')),
        'store_country_distribution': count('id').over(Window.partitionBy('country_id')),
        'store_avg_sales': avg('sale_total_price').over(window_store_sales),
    })
    .select('id', 'store_name', 'country_id', 'store_city', 'store_total_sales', 'store_city_distribution', 'store_country_distribution', 'store_avg_sales')
    .orderBy(desc('store_total_sales'), desc('store_avg_sales'), 'id')
)

In [29]:
datamart_stores.show(5)

+----+-----------+----------+----------+------------------+-----------------------+--------------------------+------------------+
|  id| store_name|country_id|store_city| store_total_sales|store_city_distribution|store_country_distribution|   store_avg_sales|
+----+-----------+----------+----------+------------------+-----------------------+--------------------------+------------------+
|1188|       DabZ|       192|    Grekan| 499.8500061035156|                      2|                        75| 499.8500061035156|
|6422|Thoughtblab|       165|     Fonte|499.79998779296875|                      1|                       332|499.79998779296875|
|6361|     Camido|       202| Longzhong|  499.760009765625|                      2|                       242|  499.760009765625|
|9923|   Edgeblab|        94|     Pesek|  499.760009765625|                      1|                      1100|  499.760009765625|
|5616|    Centizu|       165|    Tylicz| 499.7300109863281|                      2|       

In [30]:
write_to_clickhouse(datamart_stores, 'datamart_stores')

# Data Mart №5. Поставщики.
Цель: Анализ эффективности поставщиков.
- Топ-5 поставщиков с наибольшей выручкой.
- Средняя цена товаров от каждого поставщика.
- Распределение продаж по странам поставщиков.

In [31]:
dim_suppliers = base_reader_pg.option("dbtable", "dim_suppliers").load()

In [32]:
window_supplier = Window.partitionBy('id')

In [33]:
datamart_suppliers = (dim_suppliers
    .join(fact_sales
    .select('sale_supplier_id', 'sale_total_price', 'sale_quantity')
    .withColumnRenamed('sale_supplier_id', 'id'), 'id').withColumns({
        'total_revenue': sum('sale_total_price').over(window_supplier),
        'avg_product_price': avg(try_divide('sale_total_price', 'sale_quantity')).over(window_supplier),
        'country_distribution': count('id').over(Window.partitionBy('country_id')),
})).select('id', 'supplier_name', 'country_id', 'sale_total_price', 'sale_quantity', 'total_revenue', 'avg_product_price', 'country_distribution').orderBy('id')

In [34]:
datamart_suppliers.show(5)

+---+-------------+----------+----------------+-------------+------------------+------------------+--------------------+
| id|supplier_name|country_id|sale_total_price|sale_quantity|     total_revenue| avg_product_price|country_distribution|
+---+-------------+----------+----------------+-------------+------------------+------------------+--------------------+
|  1|       Tagcat|        43|           487.7|            4|487.70001220703125|121.92500305175781|                1921|
|  2|     Livetube|       165|          484.61|           10| 484.6099853515625| 48.46099853515625|                 321|
|  3|     Photobug|       179|          144.24|            9|144.24000549316406| 16.02666727701823|                   2|
|  4|        Janyx|        94|          441.09|            4| 441.0899963378906|110.27249908447266|                1079|
|  5|         Lazz|        43|           72.27|            6|  72.2699966430664|12.044999440511068|                1921|
+---+-------------+----------+--

In [35]:
write_to_clickhouse(datamart_suppliers, 'datamart_suppliers')

# Data Mart №6. Качество продукции.
Цель: Анализ отзывов и рейтингов товаров.
- Продукты с наивысшим и наименьшим рейтингом.
- Корреляция между рейтингом и объемом продаж.
- Продукты с наибольшим количеством отзывов.

In [36]:
dim_products.show(5)

+---+------------+----------------+-------------+----------------+--------------+-------------+------------+-------------+----------------+--------------------+--------------+---------------+--------------------+-------------------+-------------------+
| id|product_name|product_category|product_price|product_quantity|product_weight|product_color|product_size|product_brand|product_material| product_description|product_rating|product_reviews|product_release_date|product_expiry_date|product_supplier_id|
+---+------------+----------------+-------------+----------------+--------------+-------------+------------+-------------+----------------+--------------------+--------------+---------------+--------------------+-------------------+-------------------+
|  1|    Dog Food|            Food|        77.97|              89|          13.4|       Indigo|      Medium|        Skajo|           Steel|Aliquam quis turp...|           2.1|             97|          10/19/2011|         10/21/2028|         

In [37]:
fact_sales.show(5)

+---+----------+----------------+--------------+---------------+-------------+----------------+-------------+----------------+
| id| sale_date|sale_customer_id|sale_seller_id|sale_product_id|sale_store_id|sale_supplier_id|sale_quantity|sale_total_price|
+---+----------+----------------+--------------+---------------+-------------+----------------+-------------+----------------+
|  1| 5/14/2021|               1|             1|              1|            1|               1|            4|           487.7|
|  2|11/13/2021|               2|             2|              2|            2|               2|           10|          484.61|
|  3| 12/4/2021|               3|             3|              3|            3|               3|            9|          144.24|
|  4| 8/10/2021|               4|             4|              4|            4|               4|            4|          441.09|
|  5|  2/4/2021|               5|             5|              5|            5|               5|            6|  

In [38]:
product_sale_volume = (fact_sales
                       .groupBy('sale_product_id')
                       .agg(
                            sum('sale_quantity')
                            .alias('product_sale_volume')
                       ))

dim_products_quality = (dim_products
                        .join(
                            product_sale_volume
                            .select('sale_product_id', 'product_sale_volume')
                            .withColumnRenamed('sale_product_id', 'id'),
                        'id'))

In [39]:
datamart_product_quality = dim_products_quality.withColumns({
    'rating_volume_correlation': corr('product_rating', 'product_sale_volume').over(Window.partitionBy()),
    'total_reviews': sum('product_reviews').over(Window.partitionBy('id'))
}).select('id', 'product_name', 'product_rating', 'total_reviews', 'rating_volume_correlation', 'product_sale_volume').orderBy(desc('product_rating'), desc('total_reviews'))

In [40]:
datamart_product_quality.show(5)

+----+------------+--------------+-------------+-------------------------+-------------------+
|  id|product_name|product_rating|total_reviews|rating_volume_correlation|product_sale_volume|
+----+------------+--------------+-------------+-------------------------+-------------------+
|4845|    Dog Food|           5.0|          998|     0.001004978126268...|                  3|
|3602|    Dog Food|           5.0|          991|     0.001004978126268...|                  6|
|4084|    Dog Food|           5.0|          987|     0.001004978126268...|                  5|
|9267|   Bird Cage|           5.0|          985|     0.001004978126268...|                  8|
|2523|    Dog Food|           5.0|          966|     0.001004978126268...|                  1|
+----+------------+--------------+-------------+-------------------------+-------------------+
only showing top 5 rows



In [41]:
write_to_clickhouse(datamart_product_quality, 'datamart_product_quality')